# N'Ko ni Lonko — modèle de traduction N'Ko ↔ français (ByT5)

**Carnet modèle, à vérifier au premier lancement sur Colab.** Méthode : `docs/PLAN-LOT-2.md` ; données : `docs/DONNEES-CORPUS.md`.

Règles :
- Le corpus est **privé** : il reste sur votre **Google Drive privé**. Ne partagez jamais ce carnet avec ses résultats affichés.
- Carte conseillée : **L4 ou A100** (Colab Pro) → `byt5-base` en **bf16**. Sur la T4 gratuite : `byt5-small` en fp32. **Jamais fp16** (erreurs NaN avec T5 / ByT5).
- Colab coupe les sessions : le modèle est enregistré sur Drive régulièrement et l'entraînement **reprend** où il s'était arrêté.

In [ ]:
!pip install -q transformers datasets sacrebleu accelerate

## 1. Données (Google Drive privé)

Copier le dossier `corpus-AAAA-MM-JJ` (fabriqué par `scripts/exporter-corpus.mts`) dans `Mon Drive/nko-ni-lonko/`, puis indiquer son nom ci-dessous.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os
CORPUS = "/content/drive/MyDrive/nko-ni-lonko/corpus-AAAA-MM-JJ"   # ← à remplacer par la vraie date
SORTIE = "/content/drive/MyDrive/nko-ni-lonko/modeles/byt5-nqo-fr"
os.makedirs(SORTIE, exist_ok=True)

def lire_jsonl(chemin):
    with open(chemin, encoding="utf-8") as f:
        return [json.loads(ligne) for ligne in f if ligne.strip()]

paires = lire_jsonl(f"{CORPUS}/paires.jsonl")
lexique = lire_jsonl(f"{CORPUS}/lexique.jsonl") if os.path.exists(f"{CORPUS}/lexique.jsonl") else []
print(len(paires), "paires ·", len(lexique), "termes")

## 2. Préparation

- Seulement les paires `ok` (les `a-verifier` attendent la relecture de la rédaction).
- Longueur maximale en **octets** (une lettre N'Ko = 2 octets).
- Les deux sens dans un seul modèle, avec un préfixe : `nqo→fr : ` et `fr→nqo : `.

In [ ]:
MAX_OCTETS = 1024
GARDER_A_VERIFIER = False

def taille(t):
    return len(t.encode("utf-8"))

retenues = [p for p in paires
            if (p["controle"] == "ok" or GARDER_A_VERIFIER)
            and taille(p["nqo"]) <= MAX_OCTETS and taille(p["fr"]) <= MAX_OCTETS]
print(len(retenues), "paires retenues sur", len(paires))

def exemples(partie):
    sortie = []
    for p in retenues:
        if p["partie"] != partie:
            continue
        sortie.append({"entree": "nqo→fr : " + p["nqo"], "cible": p["fr"], "sens": "nqo→fr"})
        sortie.append({"entree": "fr→nqo : " + p["fr"], "cible": p["nqo"], "sens": "fr→nqo"})
    return sortie

from datasets import Dataset
donnees = {nom: Dataset.from_list(exemples(nom)) for nom in ["entrainement", "validation", "test"]}
for nom, d in donnees.items():
    print(nom, len(d))
# Avec peu d'articles, la partie « validation » peut être vide : on prend alors une part de l'entraînement
if len(donnees["validation"]) == 0:
    coupe = donnees["entrainement"].train_test_split(test_size=0.1, seed=42)
    donnees["entrainement"], donnees["validation"] = coupe["train"], coupe["test"]

## 3. Modèle

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODELE = "google/byt5-base" if BF16 else "google/byt5-small"
print("Carte :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucune", "· modèle :", MODELE, "· bf16 :", BF16)

tokenizer = AutoTokenizer.from_pretrained(MODELE)
modele = AutoModelForSeq2SeqLM.from_pretrained(MODELE)

def preparer(lot):
    entrees = tokenizer(lot["entree"], max_length=MAX_OCTETS + 16, truncation=True)
    cibles = tokenizer(text_target=lot["cible"], max_length=MAX_OCTETS + 16, truncation=True)
    entrees["labels"] = cibles["input_ids"]
    return entrees

jetons = {nom: d.map(preparer, batched=True, remove_columns=d.column_names) for nom, d in donnees.items()}

## 4. Entraînement (reprise automatique)

Mesure : **chrF++** (score par caractère, adapté aux langues peu dotées). Le meilleur modèle est gardé.

In [ ]:
import numpy as np, sacrebleu, glob
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

chrf = sacrebleu.metrics.CHRF(word_order=2)

def mesurer(prediction):
    sorties, etiquettes = prediction
    sorties = np.where(sorties != -100, sorties, tokenizer.pad_token_id)
    etiquettes = np.where(etiquettes != -100, etiquettes, tokenizer.pad_token_id)
    hyp = tokenizer.batch_decode(sorties, skip_special_tokens=True)
    ref = tokenizer.batch_decode(etiquettes, skip_special_tokens=True)
    return {"chrf": chrf.corpus_score(hyp, [ref]).score}

reglages = Seq2SeqTrainingArguments(
    output_dir=SORTIE,
    eval_strategy="steps", eval_steps=200,
    save_strategy="steps", save_steps=200, save_total_limit=3,
    load_best_model_at_end=True, metric_for_best_model="chrf", greater_is_better=True,
    learning_rate=3e-4, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, num_train_epochs=30, warmup_ratio=0.05,
    bf16=BF16, fp16=False,
    predict_with_generate=True, generation_max_length=MAX_OCTETS + 16,
    logging_steps=50, report_to="none",
)
entraineur = Seq2SeqTrainer(
    model=modele, args=reglages,
    train_dataset=jetons["entrainement"], eval_dataset=jetons["validation"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=modele),
    processing_class=tokenizer, compute_metrics=mesurer,
)
reprise = bool(glob.glob(f"{SORTIE}/checkpoint-*"))
entraineur.train(resume_from_checkpoint=reprise)
entraineur.save_model(f"{SORTIE}/meilleur")
tokenizer.save_pretrained(f"{SORTIE}/meilleur")

## 5. Mesure sur le test figé

chrF++ par sens, et **respect du lexique** : part des termes du dictionnaire correctement traduits (quand le terme apparaît dans la phrase source).

In [ ]:
def traduire(textes, lot=8):
    resultats = []
    for i in range(0, len(textes), lot):
        entree = tokenizer(textes[i:i + lot], return_tensors="pt", padding=True, truncation=True, max_length=MAX_OCTETS + 16).to(modele.device)
        sortie = modele.generate(**entree, max_length=MAX_OCTETS + 16, num_beams=4)
        resultats += tokenizer.batch_decode(sortie, skip_special_tokens=True)
    return resultats

test = donnees["test"]
for sens in ["nqo→fr", "fr→nqo"]:
    lot = [x for x in test if x["sens"] == sens]
    if not lot:
        continue
    hyp = traduire([x["entree"] for x in lot])
    print(sens, "chrF++ =", round(chrf.corpus_score(hyp, [[x["cible"] for x in lot]]).score, 1), f"({len(lot)} phrases)")
    if lexique:
        vus = respectes = 0
        for x, h in zip(lot, hyp):
            for t in lexique:
                src, cible = (t["termeNko"], t["termeFr"]) if sens == "nqo→fr" else (t["termeFr"], t["termeNko"])
                if src and cible and src in x["entree"]:
                    vus += 1
                    respectes += cible in h
        if vus:
            print("   respect du lexique :", respectes, "/", vus)

## 6. Comparaison avec NLLB-200 (Meta, connaît déjà le N'Ko)

⚠️ Licence **CC BY-NC** (non commerciale) : pour la **mesure** seulement. ByT5 est libre (Apache 2.0). Même test figé, sens français → N'Ko.

In [ ]:
from transformers import pipeline
nllb = pipeline("translation", model="facebook/nllb-200-distilled-600M", src_lang="fra_Latn", tgt_lang="nqo_Nkoo", device=0 if torch.cuda.is_available() else -1)
lot = [x for x in test if x["sens"] == "fr→nqo"]
hyp = [r["translation_text"] for r in nllb([x["entree"].split(" : ", 1)[1] for x in lot], max_length=512)]
print("NLLB fr→nqo chrF++ =", round(chrf.corpus_score(hyp, [[x["cible"] for x in lot]]).score, 1))

## 7. Relecture humaine

Exporter 50 traductions du test (sur le Drive privé) pour la **relecture par la rédaction**. Aucune traduction automatique n'est publiée sans cette relecture.

In [ ]:
import random, csv
random.seed(7)
lot = random.sample(list(test), min(50, len(test)))
hyp = traduire([x["entree"] for x in lot])
with open(f"{SORTIE}/relecture-50.csv", "w", encoding="utf-8", newline="") as f:
    ecrire = csv.writer(f)
    ecrire.writerow(["sens", "source", "reference", "modele", "note de 1 a 5", "remarque"])
    for x, h in zip(lot, hyp):
        ecrire.writerow([x["sens"], x["entree"].split(" : ", 1)[1], x["cible"], h, "", ""])
print("Fichier de relecture :", f"{SORTIE}/relecture-50.csv")